# IZHIKEVICH NEURON MODEL SIMULATOR

Description
-----------
This project implements the Izhikevich spiking neuron model
used widely in computational neuroscience.

The model can reproduce many biological neuron firing patterns
with minimal computational cost.

Features
--------
- neuron simulation
- spike detection
- firing rate analysis
- inter-spike interval analysis
- interactive visualizations

## 1. Import libraries

In [3]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


## 2. Input Current

In [4]:
def input_current(t):
    """
    Defines external current stimulus.

    Parameters
    ----------
    t : float
        simulation time (ms)

    Returns
    -------
    float
        injected current
    """

    if 50 <= t <= 300:
        return 10
    return 0

## 5. Spike Detection

In [5]:
def detect_spikes(voltage, threshold=30):
    """
    Detect spike indices in voltage trace.

    Parameters
    ----------
    voltage : array
        membrane voltage

    threshold : float
        spike detection threshold

    Returns
    -------
    list
        spike indices
    """

    spikes = []

    for i in range(len(voltage)):
        if voltage[i] >= threshold:
            spikes.append(i)

    return spikes

## 4. Izhikevich Neuron

In [6]:
class IzhikevichNeuron:
    """
    Implementation of the Izhikevich neuron model.

    The model reproduces diverse neuron firing behaviors
    using two coupled differential equations.
    """

    def __init__(self,
                 a=0.02,
                 b=0.2,
                 c=-65,
                 d=8,
                 dt=0.1,
                 T=400):

        """
        Initialize neuron parameters.

        Parameters
        ----------
        a,b,c,d : float
            neuron model parameters

        dt : float
            timestep (ms)

        T : float
            total simulation duration
        """

        self.a = a
        self.b = b
        self.c = c
        self.d = d

        self.dt = dt

        self.time = np.arange(0, T, dt)

        self.V = np.zeros(len(self.time))
        self.u = np.zeros(len(self.time))

        self.V[0] = -65
        self.u[0] = b * self.V[0]


    def simulate(self):
        """
        Run neuron simulation.

        Returns
        -------
        tuple
            voltage, recovery variable
        """

        for i in range(1, len(self.time)):

            V = self.V[i-1]
            u = self.u[i-1]

            I = input_current(self.time[i])

            dV = 0.04*V*V + 5*V + 140 - u + I
            du = self.a*(self.b*V - u)

            V = V + self.dt*dV
            u = u + self.dt*du

            if V >= 30:

                self.V[i] = 30
                V = self.c
                u = u + self.d

            else:

                self.V[i] = V

            self.u[i] = u

        return self.V, self.u


## 5. Inter Spike Interval Analysis

In [7]:
def compute_isi(spikes, dt):
    """
    Compute inter-spike intervals.

    Parameters
    ----------
    spikes : list
        spike indices

    dt : float
        timestep

    Returns
    -------
    array
        inter-spike intervals
    """

    spike_times = np.array(spikes)*dt
    isi = np.diff(spike_times)

    return isi

## 6. Visualization

In [8]:
def plot_results(time, V, u, spikes, isi):
    """
    Generate interactive plots.

    Visualizations include

    - membrane voltage
    - recovery variable
    - spike raster
    - ISI histogram
    """

    fig = make_subplots(
        rows=4,
        cols=1,
        subplot_titles=(
            "Membrane Voltage",
            "Recovery Variable",
            "Spike Times",
            "ISI Distribution"
        )
    )

    fig.add_trace(
        go.Scatter(x=time, y=V, name="Voltage"),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(x=time, y=u, name="Recovery Variable"),
        row=2, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=np.array(spikes)*0.1,
            y=np.ones(len(spikes)),
            mode="markers",
            name="Spikes"
        ),
        row=3, col=1
    )

    fig.add_trace(
        go.Histogram(x=isi, name="ISI"),
        row=4, col=1
    )

    fig.update_layout(
        height=900,
        title="Izhikevich Neuron Dynamics",
        template="plotly_white"
    )

    fig.show()

## 7. Main

In [9]:
def main():
    """
    Runs the full simulation and analysis pipeline.
    """

    neuron = IzhikevichNeuron()

    V, u = neuron.simulate()

    spikes = detect_spikes(V)

    isi = compute_isi(spikes, neuron.dt)

    plot_results(neuron.time, V, u, spikes, isi)


if __name__ == "__main__":
    main()